# 12 · KV cache engineering: prefill vs decode, paged cache, cross-layer sharing, windows, FP4

**Needed for:** DeepSeek-V4.1-Flash (LLM track, step 10), which is about shrinking the KV cache further.
Read the Cross-Layer Attention (Brandon et al., 2024) and YOCO (Sun et al., 2024) papers alongside.

The KV cache decides how many requests fit on a GPU and how fast decoding runs. This notebook covers the tools
that V4.1-Flash-style designs combine:

1. Why prefill and decode have different bottlenecks
2. Paged KV cache (vLLM): memory in fixed blocks, looked up through a block table
3. Cross-layer KV sharing: several layers read one layer's K/V
4. Sliding-window layers: a bounded ring buffer instead of a growing cache
5. Low-precision KV: FP4 with block scales

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"


def drill(test, fn):
    """Run a test against your own implementation; prints TODO until you write it."""
    try:
        test(fn)
    except NotImplementedError:
        print(f"TODO  {fn.__name__}")
    else:
        print(f"PASS  {fn.__name__}")

## 1. Prefill vs decode

A linear layer with a $D\times D$ bf16 weight applied to $N$ tokens does $2ND^2$ FLOPs and must read the $2D^2$
bytes of weight. Arithmetic intensity ≈ $N$ FLOPs per byte.
- **Prefill** processes the whole prompt at once ($N$ = thousands): compute-bound.
- **Decode** processes one token per sequence ($N$ = batch size): memory-bound, the GPU mostly waits on memory.
  Batching many sequences is how decoding gets efficient, and the KV cache is what limits the batch size.

That's also why some designs (like V4.1-Flash's causal encoder-decoder) use *different* parameters for the two
phases: prefill can afford more compute per token, decode can't afford more memory reads.

In [ ]:
if device == "cuda":
    from torch.utils.benchmark import Timer

    D = 4096
    lin = nn.Linear(D, D, bias=False, device=device, dtype=torch.bfloat16)
    for N in (1, 8, 64, 512, 4096):
        x = torch.randn(N, D, device=device, dtype=torch.bfloat16)
        t = Timer("lin(x)", globals={"lin": lin, "x": x}).blocked_autorange(min_run_time=0.3).median
        print(f"N={N:5d}: {t * 1e6:8.1f} us   {2 * N * D * D / t / 1e12:6.2f} TFLOP/s   {N / t / 1e3:8.1f} K tokens/s")
else:
    print("no GPU: skipping the timing demo")

Time barely changes from N = 1 to N = 64: those calls are all paying the same cost of reading the weight.

## 2. Paged KV cache

Allocating one contiguous max-length buffer per request wastes memory on space the request never uses.
vLLM's PagedAttention stores K/V in a shared pool of fixed-size **blocks** (pages). Each sequence has a
**block table**: logical block $i$ → physical block id. Blocks are allocated only as the sequence grows, and
sequences sharing a prompt prefix can point to the same physical blocks.

In [ ]:
class PagedKVCache:
    def __init__(self, n_blocks, block_size, n_heads, head_dim):
        self.k = torch.zeros(n_blocks, block_size, n_heads, head_dim)
        self.v = torch.zeros_like(self.k)
        self.block_size = block_size
        self.free = list(range(n_blocks))
        self.tables = {}   # seq_id -> list of physical block ids
        self.lengths = {}  # seq_id -> tokens stored

    def append(self, seq_id, k, v):
        """k, v: (n_heads, head_dim) for one new token."""
        table = self.tables.setdefault(seq_id, [])
        n = self.lengths.get(seq_id, 0)
        if n % self.block_size == 0:                 # current block full (or first token): grab a new one
            table.append(self.free.pop())
        blk, off = table[n // self.block_size], n % self.block_size
        self.k[blk, off], self.v[blk, off] = k, v
        self.lengths[seq_id] = n + 1

    def gather(self, seq_id):
        """Contiguous (T, n_heads, head_dim) K and V for one sequence."""
        table, n = self.tables[seq_id], self.lengths[seq_id]
        k = self.k[table].flatten(0, 1)[:n]
        v = self.v[table].flatten(0, 1)[:n]
        return k, v


def test_paged(cache_cls):
    torch.manual_seed(0)
    cache = cache_cls(n_blocks=16, block_size=4, n_heads=2, head_dim=8)
    ref = {0: [], 1: []}
    for step in range(11):                       # interleave two sequences so their blocks are not contiguous
        for sid in (0, 1):
            if sid == 1 and step > 6:
                continue
            k, v = torch.randn(2, 8), torch.randn(2, 8)
            cache.append(sid, k, v)
            ref[sid].append((k, v))
    for sid in (0, 1):
        k, v = cache.gather(sid)
        torch.testing.assert_close(k, torch.stack([a for a, _ in ref[sid]]))
        torch.testing.assert_close(v, torch.stack([b for _, b in ref[sid]]))
    assert len(cache.tables[0]) == 3 and len(cache.tables[1]) == 2   # ceil(11/4), ceil(7/4)


test_paged(PagedKVCache)
print("paged cache returns exactly what was appended")

## 3. Cross-layer KV sharing

Cross-Layer Attention (CLA): only every $s$-th layer computes K and V; the layers in between reuse them.
The cache shrinks by $s\times$ with a small quality cost. **YOCO** takes it to the limit: the first half of the
network produces one global KV cache that every layer of the second half reads.

The toy model below counts cache entries for a given sharing factor.

In [ ]:
class CLALayer(nn.Module):
    def __init__(self, d, n_heads, makes_kv):
        super().__init__()
        self.h, self.makes_kv = n_heads, makes_kv
        self.q = nn.Linear(d, d, bias=False)
        self.kv = nn.Linear(d, 2 * d, bias=False) if makes_kv else None
        self.o = nn.Linear(d, d, bias=False)

    def forward(self, x, shared_kv):
        B, T, D = x.shape
        split = lambda t: t.view(B, T, self.h, D // self.h).transpose(1, 2)
        if self.makes_kv:
            k, v = self.kv(x).chunk(2, -1)
            shared_kv = (split(k), split(v))
        a = F.scaled_dot_product_attention(split(self.q(x)), *shared_kv, is_causal=True)
        return x + self.o(a.transpose(1, 2).reshape(B, T, D)), shared_kv


def cla_cache_size(n_layers, share_every, d=64, T=32):
    layers = [CLALayer(d, 4, makes_kv=(i % share_every == 0)) for i in range(n_layers)]
    x, kv, cached = torch.randn(1, T, d), None, 0
    for layer in layers:
        x, new_kv = layer(x, kv)
        if new_kv is not kv:
            cached += sum(t.numel() for t in new_kv)
        kv = new_kv
    return cached


for s in (1, 2, 4):
    print(f"share every {s} layers: {cla_cache_size(12, s):6d} cached elements")

## 4. Sliding-window layers: a ring buffer

A layer with window $W$ never looks further back than $W$ tokens, so its cache can be a fixed $W$-slot ring
buffer: token $t$ goes in slot $t \bmod W$. Memory is bounded no matter how long the context. Attention order
doesn't matter (softmax is a sum over keys), so the scrambled slot order is fine as long as RoPE was applied
before caching.

(V4.1-Flash's "SWA bounded replay" builds on bounding the sliding-window cache like this. Read the paper for
how the replay works.)

In [ ]:
class RingKVCache:
    def __init__(self, window, n_heads, head_dim):
        self.k = torch.zeros(n_heads, window, head_dim)
        self.v = torch.zeros_like(self.k)
        self.window, self.t = window, 0

    def append(self, k, v):
        """k, v: (n_heads, head_dim)."""
        slot = self.t % self.window
        self.k[:, slot], self.v[:, slot] = k, v
        self.t += 1

    def attend(self, q):
        """q: (n_heads, head_dim) for the newest token. Attends to the last min(t, window) tokens."""
        n = min(self.t, self.window)
        return F.scaled_dot_product_attention(q[:, None], self.k[:, :n], self.v[:, :n])[:, 0]


def test_ring(cache_cls):
    torch.manual_seed(0)
    H, d, W, T = 2, 8, 4, 11
    q, k, v = torch.randn(H, T, d), torch.randn(H, T, d), torch.randn(H, T, d)
    window_mask = (torch.arange(T)[None] <= torch.arange(T)[:, None]) & (torch.arange(T)[None] > torch.arange(T)[:, None] - W)
    ref = F.scaled_dot_product_attention(q, k, v, attn_mask=window_mask)
    cache = cache_cls(W, H, d)
    for t in range(T):
        cache.append(k[:, t], v[:, t])
        torch.testing.assert_close(cache.attend(q[:, t]), ref[:, t])


test_ring(RingKVCache)
print("ring buffer decoding == sliding-window attention over the full sequence")

## 5. FP4 KV cache

Store K and V in 4-bit floats (E2M1) with a scale per 16 values, as in NVFP4 (notebook 06), and dequantize
before attention. Measure how much the attention output changes.

In [ ]:
E2M1 = torch.tensor([0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0])
E2M1 = torch.cat([-E2M1.flip(0)[:-1], E2M1])


def fp4_roundtrip(x, block=16):
    g = x.reshape(-1, block)
    scale = (g.abs().amax(-1, keepdim=True) / 6).clamp(min=1e-12).to(torch.float8_e4m3fn).float()
    q = E2M1[((g / scale).clamp(-6, 6)[..., None] - E2M1).abs().argmin(-1)]
    return (q * scale).reshape(x.shape)


q, k, v = (torch.randn(1, 8, 256, 64) for _ in range(3))
ref = F.scaled_dot_product_attention(q, k, v, is_causal=True)
for name, kk, vv in (("K and V in FP4", fp4_roundtrip(k), fp4_roundtrip(v)),
                     ("K in bf16, V in FP4", k.bfloat16().float(), fp4_roundtrip(v)),
                     ("both bf16", k.bfloat16().float(), v.bfloat16().float())):
    out = F.scaled_dot_product_attention(q, kk, vv, is_causal=True)
    print(f"{name:20s} relative output error {((out - ref).norm() / ref.norm()).item():.4f}")
print("bytes per element: bf16 2, FP4 0.5 + one FP8 scale per 16 values = 0.5625")

Plain FP4 on both K and V costs about 10% output error even on well-behaved random data. Tricks production systems
use to close the gap: keep keys (or just their RoPE dimensions) in higher precision, rotate vectors with a
Hadamard transform before quantizing to spread out outliers, and use smaller blocks. When you implement
V4.1-Flash, check which of these the paper uses and measure each one here.

## Things to be able to explain

- What limits decode throughput: FLOPs, weight reads, or KV reads? (At long context and large batch, KV reads.)
- Why does prefix sharing (same system prompt for many requests) fit naturally into a paged cache?
- What's lost with cross-layer sharing, and why might quantizing the cache be a better trade in some regimes?

## Drill

In [ ]:
class MyPagedKVCache:
    def __init__(self, n_blocks, block_size, n_heads, head_dim):
        raise NotImplementedError

    def append(self, seq_id, k, v):
        raise NotImplementedError

    def gather(self, seq_id):
        raise NotImplementedError


class MyRingKVCache:
    def __init__(self, window, n_heads, head_dim):
        raise NotImplementedError

    def append(self, k, v):
        raise NotImplementedError

    def attend(self, q):
        raise NotImplementedError


drill(test_paged, MyPagedKVCache)
drill(test_ring, MyRingKVCache)